In [9]:
import numpy as np
import pandas as pd
import time

# Đọc dữ liệu
df_ai4i = pd.read_csv("../data/raw/ai4i2020.csv")

# Lấy cột nhiệt độ
air_temp_array = np.array(df_ai4i["Air temperature [K]"])

# =========================
# 1. Tính bằng NumPy
# =========================
start = time.perf_counter()

air_temp_celsius_numpy = air_temp_array - 273.15

end = time.perf_counter()

numpy_time = end - start

print("Kết quả NumPy:", air_temp_celsius_numpy[:5])
print("Thời gian NumPy:", numpy_time, "giây")


# =========================
# 2. Tính bằng vòng lặp
# =========================
start = time.perf_counter()

air_temp_celsius_loop = []

for temp in air_temp_array:
    air_temp_celsius_loop.append(temp - 273.15)

end = time.perf_counter()

loop_time = end - start

print("Kết quả vòng lặp:", air_temp_celsius_loop[:5])
print("Thời gian vòng lặp:", loop_time, "giây")


# =========================
# 3. So sánh
# =========================
print("\n===== SO SÁNH =====")
print("Thời gian NumPy:", numpy_time)
print("Thời gian Loop:", loop_time)

if numpy_time < loop_time:
    print("NumPy nhanh hơn.")
    print("NumPy nhanh hơn khoảng", loop_time / numpy_time, "lần.")
else:
    print("Vòng lặp nhanh hơn.")


Kết quả NumPy: [24.95 25.05 24.95 25.05 25.05]
Thời gian NumPy: 0.00011409999979150598 giây
Kết quả vòng lặp: [np.float64(24.950000000000045), np.float64(25.05000000000001), np.float64(24.950000000000045), np.float64(25.05000000000001), np.float64(25.05000000000001)]
Thời gian vòng lặp: 0.02174620000005234 giây

===== SO SÁNH =====
Thời gian NumPy: 0.00011409999979150598
Thời gian Loop: 0.02174620000005234
NumPy nhanh hơn.
NumPy nhanh hơn khoảng 190.58895740393513 lần.


In [12]:
rot_speed_array = np.array(df_ai4i["Rotational speed [rpm]"])
torque_array = np.array(df_ai4i["Torque [Nm]"])
failure_array = np.array(df_ai4i["Machine failure"])

print(f"Nhiệt độ TB : {np.mean(air_temp_array):.2f} K")
print(f"Tốc độ quay max : {np.max(rot_speed_array)} rpm")
print(f"Tốc độ quay min : {np.min(rot_speed_array)} rpm")
print(f"Lực xoắn TB : {np.mean(torque_array):.2f} Nm")
print(f"Độ lệch chuẩn lực xoắn : {np.std(torque_array):.2f} Nm")

print(f"Tổng số ca hỏng : {np.sum(failure_array)} ca")
hong_nong = np.sum((air_temp_array > 300) & (failure_array == 1))
print(f"Hỏng khi nhiệt độ > 300K : {hong_nong} ca")
print(f"Tỷ lệ hỏng : {np.mean(failure_array) * 100:.2f}%")


Nhiệt độ TB : 300.00 K
Tốc độ quay max : 2886 rpm
Tốc độ quay min : 1168 rpm
Lực xoắn TB : 39.99 Nm
Độ lệch chuẩn lực xoắn : 9.97 Nm
Tổng số ca hỏng : 339 ca
Hỏng khi nhiệt độ > 300K : 230 ca
Tỷ lệ hỏng : 3.39%


In [ ]:
# ① Lọc: tốc độ quay của riêng những máy đã hỏng
speed_of_failed = rot_speed_array[failure_array == 1]
print(f"Số mẫu máy hỏng : {len(speed_of_failed)}")
print(f"Tốc độ quay TB nhóm hỏng: {np.mean(speed_of_failed):.2f} rpm")

# ② Phân vị: ngưỡng mà 95% số máy nằm dưới
torque_95th = np.percentile(torque_array, 95)
print(f"Phân vị 95 của lực xoắn : {torque_95th:.2f} Nm")

# ③ Cắt biên: ép tốc độ quay về khoảng [1200, 1500]
cleaned_speed = np.clip(rot_speed_array, 1200, 1500)
print(f"Max trước / sau np.clip : {np.max(rot_speed_array)} / {np.max(cleaned_speed)}")
print("Kết quả ép tốc độ quay về khoảng [1200, 1500]:", cleaned_speed)


Số mẫu máy hỏng : 339
Tốc độ quay TB nhóm hỏng: 1496.49 rpm
Phân vị 95 của lực xoắn : 56.10 Nm
Max trước / sau np.clip : 2886 / 1500
Kết quả ép tốc độ quay về khoảng [1200, 2000]: [1500 1408 1498 ... 1500 1408 1500]


In [7]:
df = pd.read_csv("../data/raw/ai4i2020.csv")
df.head()

print(f"Kích thước: {df.shape[0]} dòng, {df.shape[1]} cột")
df.info()
df.describe()

# Tạo cột mới — vector hóa trên cả cột, không vòng lặp
df["Air temperature [C]"] = df["Air temperature [K]"] - 273.15
print(f"Nhiệt độ không khí TB: {df['Air temperature [C]'].mean():.2f} °C")
print(f"Tổng số ca hỏng: {df['Machine failure'].sum()} ca")


Kích thước: 10000 dòng, 14 cột
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: 